# 02 — Custom CNN Training

Brain Tumor MRI Classification · Internship Project

## Setup & Imports

In [ ]:
import os, sys, random, json
import numpy as np
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

sys.path.insert(0, os.path.abspath('..'))
from src.config import (CUSTOM_CNN_PATH, EPOCHS_CUSTOM, CLASS_NAMES,
                        DISPLAY_NAMES, RESULTS_DIR, BATCH_SIZE, LR_INITIAL)
from src.preprocessing import build_generators
from src.models        import build_custom_cnn, get_callbacks
from src.evaluate      import (evaluate_model, plot_confusion_matrix,
                               plot_training_history, save_classification_report,
                               find_misclassified)
from src.data_utils    import compute_class_weights

print(f"TF version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {gpus if gpus else 'None (using CPU)'}")

## Data Generators (Custom CNN)

In [ ]:
train_gen, valid_gen, test_gen = build_generators('custom')

print("Class indices:", train_gen.class_indices)
print(f"Train batches : {len(train_gen)}")
print(f"Valid batches : {len(valid_gen)}")
print(f"Test  batches : {len(test_gen)}")

## Augmentation Visualisation

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Show original vs augmented for one image per class
fig, axes = plt.subplots(len(CLASS_NAMES), 5, figsize=(16, 13))
labels = [DISPLAY_NAMES[c] for c in CLASS_NAMES]

for row, (cls, ax_row) in enumerate(zip(CLASS_NAMES, axes)):
    folder = os.path.join(train_gen.directory, cls)
    fname  = sorted(os.listdir(folder))[0]
    fpath  = os.path.join(folder, fname)

    from PIL import Image
    orig = Image.open(fpath).convert('RGB')
    ax_row[0].imshow(orig)
    ax_row[0].set_title('Original', fontsize=9)
    ax_row[0].axis('off')
    if row == 0:
        ax_row[0].set_title('Original', fontsize=9, fontweight='bold')

    # Generate augmented versions using the training datagen
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from src.config import (AUG_ROTATION_RANGE, AUG_WIDTH_SHIFT, AUG_HEIGHT_SHIFT,
                            AUG_ZOOM_RANGE, AUG_BRIGHTNESS_RANGE,
                            AUG_HORIZONTAL_FLIP, AUG_VERTICAL_FLIP, AUG_FILL_MODE,
                            IMG_SIZE)
    aug_gen = ImageDataGenerator(
        rotation_range=AUG_ROTATION_RANGE,
        width_shift_range=AUG_WIDTH_SHIFT,
        height_shift_range=AUG_HEIGHT_SHIFT,
        zoom_range=AUG_ZOOM_RANGE,
        brightness_range=AUG_BRIGHTNESS_RANGE,
        horizontal_flip=AUG_HORIZONTAL_FLIP,
        vertical_flip=AUG_VERTICAL_FLIP,
        fill_mode=AUG_FILL_MODE,
    )
    arr = np.array(orig.resize((IMG_SIZE[1], IMG_SIZE[0]))).reshape(1, *IMG_SIZE, 3)
    it  = aug_gen.flow(arr, batch_size=1, seed=42)
    for col in range(1, 5):
        aug_arr = next(it)[0].astype(np.uint8)
        ax_row[col].imshow(aug_arr)
        ax_row[col].set_title(f'Aug {col}', fontsize=9)
        ax_row[col].axis('off')

    ax_row[0].set_ylabel(labels[row], fontsize=11, fontweight='bold',
                         rotation=90, labelpad=55)

fig.suptitle('Original vs Augmented Images (Training Data Only)', fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(RESULTS_DIR, 'augmentation_preview.png')
fig.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

## Build Custom CNN

In [ ]:
model = build_custom_cnn()
model.summary()

## Sanity Check

In [ ]:
dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
out   = model(dummy, training=False)
print(f"Output shape  : {out.shape}  (expected: (1, 4))")
print(f"Sum of probs  : {out.numpy().sum():.6f}  (expected: ~1.0)")
assert out.shape == (1, 4), "Shape mismatch!"
print("Sanity check PASSED ✓")

## Class Weights

In [ ]:
from src.config import TRAIN_DIR
cw = compute_class_weights(TRAIN_DIR, CLASS_NAMES)
print("Class weights:")
for idx, cls in enumerate(CLASS_NAMES):
    print(f"  [{idx}] {DISPLAY_NAMES[cls]:<14}: {cw[idx]:.4f}")

## Train Custom CNN

In [ ]:
callbacks = get_callbacks(CUSTOM_CNN_PATH)

history = model.fit(
    train_gen,
    epochs=EPOCHS_CUSTOM,
    validation_data=valid_gen,
    callbacks=callbacks,
    class_weight=cw,
    verbose=1,
)

## Training History Plots

In [ ]:
plot_training_history(history, 'Custom CNN')
print("Training history plots saved to results/")

## Test Set Evaluation

In [ ]:
# Reload best weights
model = tf.keras.models.load_model(CUSTOM_CNN_PATH)
result = evaluate_model(model, test_gen, 'Custom CNN')
plot_confusion_matrix(result['y_true'], result['y_pred'], 'Custom CNN')
save_classification_report(result['report'], 'Custom CNN')
find_misclassified(model, test_gen, 'Custom CNN')

## Summary

In [ ]:
print(f"Custom CNN — Test Results")
print(f"  Accuracy  : {result['accuracy']:.4f}")
print(f"  Precision : {result['precision']:.4f}")
print(f"  Recall    : {result['recall']:.4f}")
print(f"  F1-score  : {result['f1_score']:.4f}")
print(f"  Model saved: {CUSTOM_CNN_PATH}")